<a href="https://colab.research.google.com/github/galipaydin/pes517/blob/main/notebooks/lab05_data_leakage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# PES 517 · Lab 5 — How to build a model that looks brilliant and is worthless
**Artificial Intelligence Applications in Sport · Week 5**

⏱️ About 70 minutes · 🧪 This notebook creates its own data, so it will never fail to download anything

### What you will do
You will build an injury-prediction model **four times**. Each time you will remove one mistake, and each time the
performance will get worse. By the end you will have the number that is actually true — and it will be much smaller
than the one you started with.

| Stage | What we do | AUC you should get |
|---|---|---|
| 1 | The naive way everyone does it first | **≈ 0.93** |
| 2 | Find out why | — |
| 3 | Remove the leaking column | ≈ 0.80 |
| 4 | Split by athlete instead of at random | **≈ 0.69** |

> **AUC** (area under the ROC curve) runs from 0.5 (a coin flip) to 1.0 (perfect). Published injury-prediction
> models that are evaluated properly typically land around **0.73–0.77**. Keep that in mind while Stage 1 shows you 0.93.

⚠️ Run the cells in order, top to bottom. Do not skip Stage 2.


---
## Block 1 — Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, accuracy_score

RNG = np.random.default_rng(42)
print('Setup complete.')


---
## Block 2 — The data

We simulate one season for a squad of 40 athletes, 30 weeks each: **1,200 athlete-weeks**.

Because we built it, we know the truth: injury risk really does depend on the workload spike (`acwr`), soreness,
sleep and each athlete's own underlying fragility. That fragility is **not** in the data — just as in real life.

One extra column, `days_missed_next_week`, is included. It comes from the club's attendance system.
Remember that it is there.


In [ ]:
N_ATH, N_WK = 40, 30

athletes = pd.DataFrame({
    'athlete_id':     [f'A{i:02d}' for i in range(N_ATH)],
    'age':            RNG.integers(18, 34, N_ATH),
    'height_cm':      np.round(RNG.normal(180, 8, N_ATH), 1),
    'resting_hr':     RNG.integers(42, 62, N_ATH),
    'prior_injuries': RNG.poisson(1.2, N_ATH),
})
athletes['frailty'] = RNG.normal(0, 1.0, N_ATH)   # real, but never measured

rows = []
for _, a in athletes.iterrows():
    load    = RNG.normal(2600, 500, N_WK).clip(800, None)
    chronic = pd.Series(load).rolling(4, min_periods=1).mean().values
    acwr    = load / chronic
    sleep   = RNG.normal(7.4, 0.9, N_WK).clip(4, 10)
    welln   = RNG.integers(2, 6, N_WK)
    sore    = RNG.integers(1, 6, N_WK)
    lin = (-3.9 + 0.85*a['frailty'] + 1.5*(acwr - 1.0) + 0.28*sore
           - 0.22*(sleep - 7.4) + 0.18*a['prior_injuries'])
    injury = RNG.binomial(1, 1/(1 + np.exp(-lin)))
    for w in range(N_WK):
        rows.append(dict(athlete_id=a['athlete_id'], week=w+1, age=a['age'],
                         height_cm=a['height_cm'], resting_hr=a['resting_hr'],
                         prior_injuries=a['prior_injuries'],
                         weekly_load=round(load[w]), acwr=round(acwr[w], 3),
                         sleep_h=round(sleep[w], 2), wellness=welln[w],
                         soreness=sore[w], injury=int(injury[w])))

df = pd.DataFrame(rows)

# the club's attendance export: days missed in the week after this one
# not every injury costs days (some are minor); and some absences have nothing to do with injury
_from_injury = np.where(df.injury == 1,
                        RNG.integers(1, 15, len(df)) * (RNG.random(len(df)) < 0.70), 0)
_other       = (RNG.random(len(df)) < 0.05) * RNG.integers(1, 6, len(df))  # illness, personal
df['days_missed_next_week'] = np.maximum(_from_injury, _other)

print(f'{len(df)} athlete-weeks, {df.athlete_id.nunique()} athletes')
print(f'Injury rate: {df.injury.mean():.1%}')
df.head()


---
## Block 3 — The honest baseline

**Before any model, ask what a rule with no intelligence in it would score.**
If you cannot beat this, you have nothing.


In [ ]:
naive_accuracy = 1 - df.injury.mean()
print(f'Rule: "nobody will get injured this week"')
print(f'  Accuracy : {naive_accuracy:.1%}   <-- looks excellent')
print(f'  AUC      : 0.500        <-- and is completely useless')
print(f'  Injuries caught: 0 of {int(df.injury.sum())}')


> ✋ **Stop and note this.** Any model in this notebook that reports ~93% accuracy has learned nothing.
> This is why accuracy is the wrong metric for a rare outcome, and why we use AUC instead.


---
# STAGE 1 — The naive model

Throw every column at a random forest. Split the rows 75/25 at random. Report the AUC.
This is, honestly, what most first attempts look like.


In [ ]:
FEATURES_ALL = ['age','height_cm','resting_hr','prior_injuries','weekly_load',
                'acwr','sleep_h','wellness','soreness','days_missed_next_week']

X, y = df[FEATURES_ALL], df['injury']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)

m1 = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
m1.fit(Xtr, ytr)
p1 = m1.predict_proba(Xte)[:, 1]
auc1 = roc_auc_score(yte, p1)

print(f'STAGE 1 AUC = {auc1:.3f}')
print('\nThis would get published. It would also get sold to your club.')


🎉 **Congratulations, you have built a state-of-the-art injury prediction model.**

Now spend two minutes deciding whether you believe it, before you run the next cell.


---
# STAGE 2 — Find out why

Ask the model which columns it is actually using.


In [ ]:
imp = (pd.Series(m1.feature_importances_, index=FEATURES_ALL)
         .sort_values(ascending=True))

plt.figure(figsize=(7, 4.5))
colors = ['crimson' if i == 'days_missed_next_week' else 'steelblue' for i in imp.index]
plt.barh(imp.index, imp.values, color=colors)
plt.xlabel('Importance to the model')
plt.title('What is the model looking at?')
plt.tight_layout(); plt.show()

print(imp.sort_values(ascending=False).round(3))


### The problem

`days_missed_next_week` is the column doing the work. Look at what it contains:

An athlete misses days next week **because** they got injured this week. The column is not a *predictor* of injury —
it is a **consequence** of it. The model has not learned to see an injury coming. It has learned to read the
attendance record afterwards.

This is **data leakage**: information that would not be available at the moment of the real prediction has ended up
in the training data.

It is almost never as obvious as this in practice. In a real club export it arrives as `availability_status`,
`sessions_completed`, `medical_flag`, `RTP_stage` or `minutes_next_match` — all of which are quietly written
*after* the thing you are trying to predict.


In [ ]:
print(df.groupby('injury')['days_missed_next_week'].describe()[['count','mean','max']].round(2))


---
# STAGE 3 — Remove the leaking column

Same model, same random split, one column dropped.


In [ ]:
FEATURES_CLEAN = [f for f in FEATURES_ALL if f != 'days_missed_next_week']

X, y = df[FEATURES_CLEAN], df['injury']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=.25, random_state=0, stratify=y)

m3 = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
m3.fit(Xtr, ytr)
auc3 = roc_auc_score(yte, m3.predict_proba(Xte)[:, 1])

print(f'STAGE 1 AUC (leaky) = {auc1:.3f}')
print(f'STAGE 3 AUC (clean) = {auc3:.3f}')
print(f'Cost of one leaking column: {auc1 - auc3:.3f} AUC')


### But this is still not honest

We split **1,200 rows** at random. Those rows are not 1,200 independent observations — they are 40 athletes
observed 30 times each.

A random split puts *the same athlete* in both the training set and the test set. The model does not have to learn
anything about injury risk in general; it can learn that **athlete A17 is fragile**, and then recognise A17 in the
test set from their height, resting heart rate and age.

That is not prediction. That is recognition. And it tells you nothing about the new signing arriving in January.


---
# STAGE 4 — Split by athlete

`GroupShuffleSplit` guarantees that every athlete is entirely in the training set or entirely in the test set — never both.


In [ ]:
groups = df['athlete_id']
gss = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=0)
tr_idx, te_idx = next(gss.split(X, y, groups))

print(f'Athletes in training: {groups.iloc[tr_idx].nunique()}')
print(f'Athletes in test    : {groups.iloc[te_idx].nunique()}')
print(f'Overlap             : {len(set(groups.iloc[tr_idx]) & set(groups.iloc[te_idx]))} athletes\n')

m4 = RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=0, n_jobs=-1)
m4.fit(X.iloc[tr_idx], y.iloc[tr_idx])
p4 = m4.predict_proba(X.iloc[te_idx])[:, 1]
auc4 = roc_auc_score(y.iloc[te_idx], p4)

print(f'STAGE 4 AUC (clean + grouped by athlete) = {auc4:.3f}')


In [ ]:
stages = ['1. Leaky\n(random split)', '3. Clean\n(random split)',
          '4. Clean\n(grouped split)', 'Coin flip']
vals   = [auc1, auc3, auc4, 0.5]

plt.figure(figsize=(7.5, 4.5))
bars = plt.bar(stages, vals, color=['crimson', 'orange', 'seagreen', 'lightgrey'])
plt.axhline(0.5, ls='--', c='grey')
plt.axhspan(0.73, 0.77, alpha=.15, color='navy',
            label='typical honest range in the published literature')
plt.ylim(0.4, 1.0); plt.ylabel('AUC')
plt.title('The same data, the same model, four different truths')
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width()/2, v + .012, f'{v:.3f}', ha='center', fontweight='bold')
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout(); plt.show()


---
## Block 6 — Break it 💥

One more, for the road. Change `TEST_SIZE` below to `0.02` and re-run: the test set now contains about
24 athlete-weeks and roughly 2 injuries. Watch the AUC swing wildly between random seeds.

**A number computed on a tiny test set is noise with a decimal point.**


In [ ]:
TEST_SIZE = 0.02          # ✏️ EDIT ME — try 0.02, then 0.10, then 0.25

aucs = []
for seed in range(8):
    gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=seed)
    tr, te = next(gss.split(X, y, groups))
    if y.iloc[te].nunique() < 2:
        continue
    mm = RandomForestClassifier(n_estimators=200, min_samples_leaf=2, random_state=0, n_jobs=-1)
    mm.fit(X.iloc[tr], y.iloc[tr])
    aucs.append(roc_auc_score(y.iloc[te], mm.predict_proba(X.iloc[te])[:, 1]))

if len(aucs) < 2:
    print('The test set was so small that most splits contained no injuries at all.')
    print('That is itself the lesson: raise TEST_SIZE and try again.')
else:
    print(f'test_size = {TEST_SIZE}')
    print('AUC across 8 random seeds:', [round(a, 3) for a in aucs])
    print(f'range = {max(aucs) - min(aucs):.3f}   <-- the same model, the same data')


---
## Block 7 — Interpretation ✍️ **(this is what is graded)**

Your club bought the Stage 1 model. It was demonstrated to the board at **AUC 0.93** and it is now on a screen
in the medical room, flagging athletes as high risk.

You have just run this notebook.

**Write the email you send to the Performance Director.** About 200 words. It must:

1. Say what is wrong, in language a non-technical director will follow — no jargon, or jargon you define in the same sentence.
2. State what the model's real performance is, and what that means in practice (out of 100 flagged athletes, how many are you actually catching?).
3. Say what you recommend doing on Monday morning — you may not simply say "switch it off", because the medical staff have started relying on it.
4. State the one question you would have asked the vendor before purchase that would have caught this.

---

**Your email:**

*(replace this line)*


---
## Block 8 — AI Use Log

| Tool | What I asked it | What it gave me | What I changed or rejected |
|---|---|---|---|
|  |  |  |  |


---
## What to take out of this lab

1. **Always compute the trivial baseline first.** 93% accuracy meant nothing here.
2. **Ask what each column means and when it was written.** Anything recorded after the outcome is leakage.
3. **Repeated measurements on the same people are not independent rows.** Split by athlete, not by row.
4. **A high number is a reason for suspicion, not celebration.** If an injury model beats AUC 0.80, find out why before you believe it.

You will apply all four of these to your own term project, where 25 of the 100 marks are for exactly this.
